In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Project root handling (works whether cwd is Thesis/ or Thesis/scripts/)
CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "scripts" else CWD
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import finite_difference_quantum as fdq
from scipy.sparse.linalg import ArpackNoConvergence, eigs, eigsh

plt.rcParams.update(
    {
        "font.family": "serif",
        "mathtext.fontset": "stix",
        "axes.linewidth": 1.1,
        "grid.alpha": 0.25,
        "grid.linestyle": "--",
        "grid.linewidth": 0.8,
        "figure.dpi": 300,
        "savefig.dpi": 300,
    }
)


In [2]:
def analytical_isw_eigenvalues(n_max: int, L: float) -> np.ndarray:
    n = np.arange(1, int(n_max) + 1)
    return (n**2) * (np.pi**2) / (2.0 * (float(L) ** 2))


def numerical_isw_eigenvalues(*, N: int, order: int, k: int, L: float) -> np.ndarray:
    """Compute the lowest k ISW eigenvalues using your Hamiltonian.

    Important solver detail:
    - order=2: use `eigsh` (Hermitian ARPACK)
    - order=4: your boundary closures make H slightly non-symmetric, so `eigsh`
      can fail. Use `eigs` (general ARPACK) with shift-invert targeting small
      eigenvalues near sigma=0.
    """

    def V0(x):
        return np.zeros_like(x)

    H = fdq.hamiltonian(V0, 0.0, float(L), int(N), order=int(order))

    if int(order) == 2:
        ev = eigsh(
            H,
            k=int(k),
            which="SA",
            tol=1e-10,
            maxiter=200000,
            return_eigenvectors=False,
        )
        ev = np.real(ev)
        return np.sort(ev)

    # order=4: shift-invert around 0 to target smallest eigenvalues
    try:
        ev = eigs(
            H,
            k=int(k),
            sigma=0.0,
            which="LM",
            tol=1e-10,
            maxiter=400000,
            return_eigenvectors=False,
        )
    except ArpackNoConvergence as e:
        if getattr(e, "eigenvalues", None) is not None and len(e.eigenvalues) >= int(k):
            ev = e.eigenvalues[: int(k)]
        elif getattr(e, "eigenvalues", None) is not None and len(e.eigenvalues) > 0:
            # If partial convergence happened, use what we got.
            ev = e.eigenvalues
        else:
            raise

    ev = np.real(np.asarray(ev))
    ev = np.sort(ev)
    # If partial convergence gave fewer values, still return what we have.
    return ev


In [3]:
# ---- Configuration (edit these) ----
L = 10.0
N = 6000
n_max = 60

out_path = (PROJECT_ROOT / "images" / "isw_numerical_vs_analytical_eigenvalues.png").resolve()
print("Will write:", out_path)
print("Using L=", L, " N=", N, " n_max=", n_max)


Will write: /Users/joshhiller/Desktop/Thesis/images/isw_numerical_vs_analytical_eigenvalues.png
Using L= 10.0  N= 6000  n_max= 60


In [4]:
E_exact = analytical_isw_eigenvalues(n_max=n_max, L=L)

# progress prints are intentionally small (nbconvert can be slow with lots of stdout)
print("Computing numerical eigenvalues...")
E2 = numerical_isw_eigenvalues(N=N, order=2, k=n_max, L=L)
E4 = numerical_isw_eigenvalues(N=N, order=4, k=n_max, L=L)

# If order=4 partially converged, trim everything consistently.
k_eff = min(len(E_exact), len(E2), len(E4))
E_exact = E_exact[:k_eff]
E2 = E2[:k_eff]
E4 = E4[:k_eff]
n = np.arange(1, k_eff + 1)

rel2 = np.abs((E2 - E_exact) / E_exact)
rel4 = np.abs((E4 - E_exact) / E_exact)

print("k_eff:", k_eff)


Computing numerical eigenvalues...


k_eff: 60


In [5]:
fig = plt.figure(figsize=(11.0, 4.0))
gs = fig.add_gridspec(1, 2, width_ratios=[1.35, 1.0], wspace=0.28)
ax0 = fig.add_subplot(gs[0, 0])
ax1 = fig.add_subplot(gs[0, 1])

# Fewer x tick labels so the axis stays readable
n_max_plot = int(n.max())
tick_step = 5 if n_max_plot <= 80 else 10
xticks = np.arange(1, n_max_plot + 1, tick_step)

# Smaller markers (requested)
ms_scatter = 3.0
ms_line = 2.6

# Left: eigenvalues
ax0.plot(n, E_exact, "-", color="#222222", linewidth=2.0, label="Analytical")
ax0.plot(n - 0.10, E2, "o", color="#1f77b4", markersize=ms_scatter, label="2nd order (FD)")
ax0.plot(n + 0.10, E4, "s", color="#ff7f0e", markersize=ms_scatter, label="4th order (FD)")
ax0.set_xlabel(r"Quantum number $n$")
ax0.set_ylabel(r"Energy $E_n$")
ax0.set_xticks(xticks)
ax0.grid(True, alpha=0.18, linestyle="--", linewidth=0.8)
ax0.legend(frameon=False, fontsize=9, loc="upper left")
ax0.set_title(rf"Eigenvalues (N={N})", fontsize=12, pad=8)

# Right: relative error (log scale)
eps = 1e-20
ax1.semilogy(
    n,
    np.maximum(rel2, eps),
    "o-",
    color="#1f77b4",
    linewidth=1.6,
    markersize=ms_line,
    label="2nd order",
    markevery=2,
)
ax1.semilogy(
    n,
    np.maximum(rel4, eps),
    "s--",
    color="#ff7f0e",
    linewidth=1.6,
    markersize=ms_line,
    label="4th order",
    markevery=2,
)
ax1.set_xlabel(r"Quantum number $n$")
ax1.set_ylabel(r"Relative error $|E_n - E_n^{\mathrm{exact}}|/E_n^{\mathrm{exact}}$")
ax1.set_xticks(xticks)
ax1.grid(True, which="both", alpha=0.18, linestyle="--", linewidth=0.8)
ax1.legend(frameon=False, fontsize=9, loc="upper left")
ax1.set_title("Relative error (log scale)", fontsize=12, pad=8)

fig.suptitle("ISW benchmark: numerical vs analytical eigenvalues", fontsize=13, y=1.02)

out_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_path, bbox_inches="tight")
plt.close(fig)

print("Saved:", out_path)


Saved: /Users/joshhiller/Desktop/Thesis/images/isw_numerical_vs_analytical_eigenvalues.png
